In [38]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 0, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 23, 59))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 207.24it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,1563618455000000301,NaN,NEWT,TRAD,2025-12-29 05:01:14+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
1,1563618456000000401,1.563618e+18,MODI,TRAD,2025-12-29 05:01:16+00:00,True,IR,None,I,True,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
2,1563618457000000501,1.563618e+18,MODI,TRAD,2025-12-29 05:01:19+00:00,False,IR,None,I,True,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
3,1563618458000000601,1.563618e+18,MODI,TRAD,2025-12-29 05:01:23+00:00,False,IR,None,I,True,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
4,1563618910000000101,1.563616e+18,CORR,,2025-12-29 05:01:29+00:00,None,IR,None,I,True,...,,3.0,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11173,1575253883000000101,NaN,NEWT,TRAD,2025-12-30 04:56:51+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZ4HTNF4RVB5,NA/Swap Flt Flt AUD USD,AUD-BBSW vs USD-SOFR-OIS Compound
11174,1575167371000000101,NaN,NEWT,TRAD,2025-12-30 04:57:00+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZXF4PFZSLP7,NA/Swap OIS INR,INR-MIBOR-OIS Compound
11175,1575159075000000101,NaN,NEWT,TRAD,2025-12-30 04:57:10+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZV61TRS4HD9,NA/Swap OIS JPY,JPY-TONA-OIS-COMPOUND
11176,1575168481000000101,NaN,NEWT,TRAD,2025-12-30 04:57:34+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZ26BPG38C1K,NA/Swap Fxd Flt KRW,KRW-CD 91D


In [40]:
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions

sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf

Classifying Trades: 100%|██████████| 392/392 [00:00<00:00, 1674.00trade/s]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,estimated_pv01,...,underlying_expiration_date,tenor_years,tenor_label,forward_start_years,forward_label,premium,exercise_style,strike,is_capped,Dissemination Identifier
143,NEWT-TRAD,1568561409000000201,2025-12-29 09:17:55+00:00,2025-12-29,2026-12-29,UNKNOWN,USD-SOFR-OIS Compound 1D CONSTANT 1Y6Y CHOOSER...,1.000000e+07,USD,0.0,...,2032-12-31,6.094444,6Y,1.013889,1Y,164810.0,EUROPEAN,0.03576,False,1568561409000000201
142,NEWT-NOVA,1568080997000000501,2025-12-29 09:19:10+00:00,2025-12-18,2026-08-21,SWAPTION_PAYER,USD-SOFR-OIS Compound 1Y CONSTANT 8M1Y PAYER E...,1.000000e+09,USD,0.0,...,2027-08-25,1.025000,1Y,0.683333,8M,0.0,EUROPEAN,0.03250,False,1568080997000000501
157,NEWT-TRAD,1570738481000000101,2025-12-29 13:05:00+00:00,2025-12-29,2026-03-30,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 3M10Y PAYER EURO...,1.000000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,0.0,EUROPEAN,0.03760,False,1570738481000000101
156,NEWT-TRAD,1570737966000000101,2025-12-29 13:05:00+00:00,2025-12-29,2026-02-27,SWAPTION_RECEIVER,USD-SOFR-COMPOUND 1D CONSTANT 2M10Y RECEIVER E...,2.000000e+08,USD,0.0,...,2036-03-03,10.158333,10Y,0.166667,2M,0.0,EUROPEAN,0.03755,False,1570737966000000101
155,NEWT-TRAD,1570737004000000101,2025-12-29 13:05:00+00:00,2025-12-29,2026-03-30,SWAPTION_RECEIVER,USD-SOFR-COMPOUND 1D CONSTANT 3M10Y RECEIVER E...,1.000000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,2300000.0,EUROPEAN,0.03760,False,1570737004000000101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,CORR-,1573690155000000101,2025-12-29 21:57:16+00:00,2025-12-29,2026-01-30,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1M30Y PAYER ...,1.000000e+07,USD,0.0,...,2056-02-02,30.444444,30Y,0.088889,1M,122500.0,EUROPEAN,0.04123,False,1573690155000000101
61,CORR-,1573695808000000101,2025-12-29 21:58:17+00:00,2025-12-29,2026-01-30,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1M30Y RECEIV...,1.000000e+07,USD,0.0,...,2056-02-02,30.444444,30Y,0.088889,1M,122500.0,EUROPEAN,0.04123,False,1573695808000000101
70,TERM-ETRM,1573785086000000101,2025-12-29 22:12:24+00:00,2025-12-29,2026-03-30,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3M5Y RECEIVE...,3.000000e+08,USD,0.0,...,2031-04-01,5.077778,5Y,0.252778,3M,0.0,EUROPEAN,0.03420,False,1573785086000000101
384,NEWT-TRAD,1584543272000000201,2025-12-30 01:57:19+00:00,2025-12-08,2026-01-12,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1M30Y PAYER ...,6.000000e+07,USD,0.0,...,2056-01-14,30.441667,30Y,0.097222,1M,1297200.0,EUROPEAN,0.04110,False,1584543272000000201


In [43]:
temp = df[df["Dissemination Identifier"].isin(sdf["trade_id"])]
temp["Execution Timestamp"] = temp["Execution Timestamp"].astype(str)
temp["Event timestamp"] = temp["Event timestamp"].astype(str)
temp.to_excel("swaption_trades.xlsx",index=False)

C:\Users\chris\AppData\Local\Temp\ipykernel_79052\2697270985.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\chris\AppData\Local\Temp\ipykernel_79052\2697270985.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [32]:
sdf[sdf["is_capped"] == True]

# sdf["trade_label"].value_counts().head(10)
# sdf[sdf["trade_label"] == "USD-SOFR-OIS Compound 1D CONSTANT 9Y10Y PAYER EURO VANILLA PHYS"]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,estimated_pv01,...,underlying_expiration_date,tenor_years,tenor_label,forward_start_years,forward_label,premium,exercise_style,strike,is_capped,Dissemination Identifier
169,NEWT-TRAD,1570783121000001401,2025-12-29 13:52:52+00:00,2025-12-29,2028-12-28,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Y1Y RECEIVE...,6.500000e+08,USD,0.0,...,2030-01-02,1.027778,1Y,3.041667,3Y,0.000,EUROPEAN,0.035130,True,1570783121000001401
168,NEWT-TRAD,1570783120000001301,2025-12-29 13:52:52+00:00,2025-12-29,2028-12-28,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Y1Y RECEIVE...,6.500000e+08,USD,0.0,...,2030-01-02,1.027778,1Y,3.041667,3Y,7540000.000,EUROPEAN,0.035130,True,1570783120000001301
202,TERM-EXER,1570979154000000501,2025-12-29 16:00:27+00:00,2025-10-28,2025-12-29,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 2M10Y PAYER ...,2.500000e+08,USD,0.0,...,2035-12-31,10.150000,10Y,0.172222,2M,0.000,EUROPEAN,0.037175,True,1570979154000000501
0,TERM-EXER,1570966806000001101,2025-12-29 16:00:29+00:00,2025-11-20,2025-12-29,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1M10Y PAYER ...,2.500000e+08,USD,0.0,...,2035-12-31,10.150000,10Y,0.108333,1M,34375.000,EUROPEAN,0.035675,True,1570966806000001101
44,MODI-TRAD,1570991358000000101,2025-12-29 16:18:01+00:00,2025-12-29,2028-12-29,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Y1M1Y RECEI...,6.500000e+08,USD,0.0,...,2030-01-03,1.027778,1Y,3.044444,3Y1M,7540000.000,EUROPEAN,0.035130,True,1570991358000000101
45,MODI-TRAD,1570991359000000201,2025-12-29 16:18:02+00:00,2025-12-29,2028-12-29,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Y1M1Y RECEI...,6.500000e+08,USD,0.0,...,2030-01-03,1.027778,1Y,3.044444,3Y1M,0.000,EUROPEAN,0.035130,True,1570991359000000201
247,NEWT-TRAD,1572479164000000101,2025-12-29 18:20:49+00:00,2025-12-29,2026-03-30,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3M10Y RECEIV...,2.500000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,1043753.340,EUROPEAN,0.035150,True,1572479164000000101
248,NEWT-TRAD,1572479167000000401,2025-12-29 18:20:56+00:00,2025-12-29,2026-03-30,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 3M10Y PAYER ...,2.500000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,1056253.380,EUROPEAN,0.040150,True,1572479167000000401
88,CORR-,1572504398000000301,2025-12-29 18:33:43+00:00,2025-12-29,2026-03-30,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3M10Y RECEIV...,2.500000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,1056253.380,EUROPEAN,0.040150,True,1572504398000000301
87,CORR-,1572504985000000101,2025-12-29 18:33:45+00:00,2025-12-29,2026-03-30,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 3M10Y PAYER ...,2.500000e+08,USD,0.0,...,2036-04-01,10.152778,10Y,0.252778,3M,1043753.340,EUROPEAN,0.035150,True,1572504985000000101


In [31]:
sdf.iloc[0].to_dict()

{'event_action': 'NEWT-TRAD',
 'trade_id': 1568561409000000201,
 'execution_timestamp': Timestamp('2025-12-29 09:17:55+0000', tz='UTC'),
 'effective_date': Timestamp('2025-12-29 00:00:00'),
 'expiration_date': Timestamp('2026-12-29 00:00:00'),
 'product_type': 'UNKNOWN',
 'trade_label': 'USD-SOFR-OIS Compound 1D CONSTANT 1Y6Y CHOOSER EURO VANILLA ELECT AT EXERCISE',
 'notional': 10000000.0,
 'notional_currency': 'USD',
 'estimated_pv01': 0.0,
 'package_type': 'SWAPTION',
 'package_id': None,
 'package_legs': None,
 'underlying_expiration_date': Timestamp('2032-12-31 00:00:00'),
 'tenor_years': 6.094444444444444,
 'tenor_label': '6Y',
 'forward_start_years': 1.0138888888888888,
 'forward_label': '1Y',
 'premium': 164810.0,
 'exercise_style': 'EUROPEAN',
 'strike': 0.03576,
 'is_capped': False,
 'Dissemination Identifier': 1568561409000000201}

In [30]:
df[df["Dissemination Identifier"] == 1572501480000000201].iloc[0].to_dict()

{'Dissemination Identifier': 1572501480000000201,
 'Original Dissemination Identifier': 1725643037.0,
 'Action type': 'TERM',
 'Event type': 'ETRM',
 'Event timestamp': Timestamp('2025-12-29 18:30:28+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'N',
 'Mandatory clearing indicator': False,
 'Execution Timestamp': Timestamp('2025-09-16 14:54:31+0000', tz='UTC'),
 'Effective Date': Timestamp('2025-09-16 00:00:00'),
 'Expiration Date': Timestamp('2026-06-24 00:00:00'),
 'Maturity date of the underlier': datetime.date(2028, 6, 26),
 'Non-standardized term indicator': False,
 'Platform identifier': 'BILT',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': False,
 'Large notional off-facility swap election indicator': True,
 'Notional amount-Leg 1': '5',
 'Notional amount-Leg 2': '5',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': 'USD',
 'Notional quantity-Leg 1': None,
 'Notional qu

In [ ]:
# from SDRUtils.products._swaptions.upi import make_swaption_desc_func, _build_upi_df

# swaption_upis = _build_upi_df()

# mask = df["Unique Product Identifier"].isin(swaption_upis["swaption_Identifier_UPI"])
# swaption_trades_df = df.loc[mask].copy()

# desc_fn = make_swaption_desc_func()

# swaption_trades_df.loc[:, "description"] = swaption_trades_df.apply(desc_fn, axis=1)
# swaption_trades_df

In [11]:
temp = sdf 
# temp["Execution sf"] = temp["Execution Timestamp"].astype(str)
# temp["Event timestamp"] = temp["Event timestamp"].astype(str)
temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
temp.to_excel("swaption_trades2.xlsx",index=False)

In [ ]:
swaption QZZGWPNBF5R3 USD CALL Euro Vanilla Phys, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys
swaption QZNLQ8T0N0SX USD PUTO Euro Vanilla Phys, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys

swaption QZMMWR8JKZQ8 USD PUTO Euro Vanilla Phys, underlying QZXG4P1G2KCS USD-SOFR-OIS Compound 1D Constant Phys 
swaption QZWXKVHB5F8V USD CALL Euro Vanilla Phys, underlying QZXG4P1G2KCS USD-SOFR-OIS Compound 1D Constant Phys 

swaption QZMRJ6051HQB USD OPTL Euro Vanilla Phys, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys 

swaption QZ7B7ZPS1LS5 USD PUTO Euro Vanilla Phys, underlying QZ8RQJKXHZP9 USD-SOFR-OIS Compound 1D Constant Cash

swaption QZXZSN00ZVCG USD PUTO Euro Vanilla Phys, underlying QZ1CXH05JJJH USD-SOFR-COMPOUND 1D Constant PHYS 
swaption QZXSN072GFF3 USD CALL Euro Vanilla Phys, underlying QZ1CXH05JJJH USD-SOFR-COMPOUND 1D Constant PHYS

swaption QZZLNQ2D4JQT USD CALL Euro Vanilla Phys, underlying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS
swaption QZJ92TTHTSF0 USD PUTO Euro Vanilla Phys, underyying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS
swaption QZHQPRHC3S7T USD OPTL Euro Vanilla Phys, underlying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS
swaption QZJ7QTJM4H97 USD PUTO Euro Vanilla Cash, underlying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS
swaption QZVLJBR2Z1VS USD Call Euro Vanilla Cash, underlying QZKQ8QSWZKR2 USD-SOFR-OIS Compound 1Y Constant PHYS

swaption QZKDXXXPW3X3 USD OPTL Euro Vanilla Phys, underlying QZ749DHWD023 USD-SOFR-COMPOUND 1D Constant Cash

swaption QZJ7PRWFV1G1 USD PUTO Euro Vanilla Cash, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys
swaption QZ1SC5570DC7 USD CALL Euro Vanilla Cash, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys
swaption QZJBL64PCV75 USD OPTL Euro Vanilla OPTL, underlying QZPB5VSBGRCD USD-SOFR-OIS Compound 1D Constant Phys

swaption QZ7XW3KWNFQ0 USD CALL BERM Vanilla Cash, underlying QZWVFGTL0HX0 USD-SOFR-COMPOUND 1D Constant Cash
swaption QZP4BJ4T4W9T USD CALL Euro Vanilla Cash, underlying QZWVFGTL0HX0 USD-SOFR-COMPOUND 1D Constant Cash

swaption QZC8L2PMNJZH USD PUTO Euro Vanilla Phys, underlying QZM88X1WWFMD USD-SOFR 1D Constant PHYS

swaption QZT2NCB2NRFJ USD Call Euro Vanilla Phys, underlying QZFDML7GL67S USD-LIBOR-BBA 3M Constant Cash

swaption QZRP1RVPFLQT USD PUTO Euro Vanilla Cash, underlying QZ5JTCF06XV6 USD-SOFR 1D Custom Cash
swaption QZX5JQN0TK24 USD Call Euro Vanilla Cash, underlying QZ5JTCF06XV6 USD-SOFR 1D Custom Cash

swaption QZBK1N7NJ0VF USD Call Euro Vanilla Phys, underlying QZ749DHWD023 USD-SOFR-COMPOUND 1D Constant Cash
swaption QZ2GZS235CHW USD PUTO Euro Vanilla Phys, underlying QZ749DHWD023 USD-SOFR-COMPOUND 1D Constant Cash

swaption QZTRSNZTX0F0 USD PUTO Berm Vanilla Phys, underlying QZPFD1BKD6MW USD-SOFR CME Term 1M Accreting Phys

swaption QZMC010F38PK USD PUTO Euro Vanilla Phys, underlying QZ1CMVCKP2QW USD-SOFR 3M Constant Phys
swaption QZVQ21ZLNJ5C USD OPTL Euro Vanilla Phys, underlying QZ1CMVCKP2QW USD-SOFR 3M Constant Phys
swaption QZJ03ZN7VKG7 USD Call Euro Vanilla Cash, underlying QZ1CMVCKP2QW USD-SOFR 3M Constant Phys 


In [128]:
# import datetime
# from gs_quant.data import Dataset
# from gs_quant.session import GsSession

# gs_client_id = "2eb2f48872304c1d94fa1642fa691afe"
# gs_secret_key = "91cb9c89110495d1f62d0ab0c4014555c992c2509de8f5ae2b8bf1a2d3c86bd4"
# GsSession.use(client_id=gs_client_id, client_secret=gs_secret_key, scopes=('read_product_data',))

# usd_sofr_1y10y_asset_id = "MA3YAP9YTBN0HAM4"

# start = datetime.date(2025, 1, 1)
# end = datetime.date(2025, 11, 20)

# df = Dataset("IR_SWAPTION_VOLS_V1_STANDARD").get_data(start=start, end=end, assetId=usd_sofr_1y10y_asset_id)

# df["bpvol_yr"] = df["impliedNormalVolatility"] * (252 ** (0.5))

# df

